# Visibility Analysis

This notebook demonstrates multi-observer viewshed analysis and point-to-point
line-of-sight profiling.

**Functions covered:**
- `cumulative_viewshed` -- count of observers with line-of-sight to each cell
- `visibility_frequency` -- normalized version (fraction of observers)
- `line_of_sight` -- elevation profile, visibility, and optional Fresnel zone clearance

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.terrain import generate_terrain
from xrspatial.hillshade import hillshade
from xrspatial.visibility import (
    cumulative_viewshed,
    visibility_frequency,
    line_of_sight,
)

## Generate synthetic terrain

In [ ]:
terrain = generate_terrain(width=200, height=200, seed=42)
hs = hillshade(terrain)

fig, ax = plt.subplots(figsize=(8, 8))
hs.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, alpha=0.4, cmap='terrain', add_colorbar=True)
ax.set_title('Synthetic terrain')
plt.tight_layout()

## Cumulative viewshed

Place three observers on the terrain and count how many can see each cell.

In [ ]:
xs = terrain.coords['x'].values
ys = terrain.coords['y'].values

observers = [
    {'x': float(xs[40]),  'y': float(ys[40]),  'observer_elev': 20},
    {'x': float(xs[160]), 'y': float(ys[80]),  'observer_elev': 20},
    {'x': float(xs[100]), 'y': float(ys[160]), 'observer_elev': 20},
]

cum = cumulative_viewshed(terrain, observers)

fig, ax = plt.subplots(figsize=(8, 8))
hs.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
cum.plot.imshow(ax=ax, alpha=0.6, cmap='YlOrRd',
                add_colorbar=True, vmin=0, vmax=3)
for obs in observers:
    ax.plot(obs['x'], obs['y'], 'k^', markersize=10)
ax.set_title('Cumulative viewshed (observer count)')
plt.tight_layout()

## Visibility frequency

Same data, but normalized to [0, 1].

In [ ]:
freq = visibility_frequency(terrain, observers)

fig, ax = plt.subplots(figsize=(8, 8))
hs.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
freq.plot.imshow(ax=ax, alpha=0.6, cmap='RdYlGn',
                 add_colorbar=True, vmin=0, vmax=1)
for obs in observers:
    ax.plot(obs['x'], obs['y'], 'k^', markersize=10)
ax.set_title('Visibility frequency')
plt.tight_layout()

## Line of sight

Draw a transect between two points and check visibility along it.

In [ ]:
ox, oy = float(xs[20]), float(ys[100])
tx, ty = float(xs[180]), float(ys[100])

los = line_of_sight(terrain, x0=ox, y0=oy, x1=tx, y1=ty,
                    observer_elev=15, target_elev=5)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(los['distance'].values, 0, los['elevation'].values,
                color='sienna', alpha=0.4, label='Terrain')
ax.plot(los['distance'].values, los['los_height'].values,
        'r--', label='LOS ray')

vis = los['visible'].values
d = los['distance'].values
ax.scatter(d[vis], los['elevation'].values[vis],
           c='green', s=8, label='Visible', zorder=3)
ax.scatter(d[~vis], los['elevation'].values[~vis],
           c='red', s=8, label='Blocked', zorder=3)

ax.set_xlabel('Distance')
ax.set_ylabel('Elevation')
ax.set_title('Line-of-sight profile')
ax.legend()
plt.tight_layout()

## Fresnel zone clearance

For radio link planning, check whether the first Fresnel zone is clear at 900 MHz.

In [ ]:
los_f = line_of_sight(terrain, x0=ox, y0=oy, x1=tx, y1=ty,
                      observer_elev=50, target_elev=50,
                      frequency_mhz=900)

fig, ax = plt.subplots(figsize=(12, 4))
d = los_f['distance'].values
ax.fill_between(d, 0, los_f['elevation'].values,
                color='sienna', alpha=0.4, label='Terrain')
ax.plot(d, los_f['los_height'].values, 'r--', label='LOS ray')

# Fresnel zone envelope
lh = los_f['los_height'].values
fr = los_f['fresnel_radius'].values
ax.fill_between(d, lh - fr, lh + fr, alpha=0.15, color='blue',
                label='1st Fresnel zone')

fc = los_f['fresnel_clear'].values
ax.scatter(d[fc], los_f['elevation'].values[fc],
           c='green', s=8, label='Fresnel clear', zorder=3)
ax.scatter(d[~fc], los_f['elevation'].values[~fc],
           c='red', s=8, label='Fresnel blocked', zorder=3)

ax.set_xlabel('Distance')
ax.set_ylabel('Elevation')
ax.set_title('Fresnel zone clearance (900 MHz)')
ax.legend()
plt.tight_layout()